In [6]:
# ============================================================
# 0) Install + imports
# ============================================================
!pip -q install lifelines

import os
import random
import copy
import numpy as np
import pandas as pd

from lifelines import CoxPHFitter


# ============================================================
# 1) Reproducibility
# ============================================================
RNG_SEED = 42

def seed_all(seed: int = RNG_SEED) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

seed_all(RNG_SEED)


# ============================================================
# 2) Load data
# ============================================================
url = "https://ndownloader.figshare.com/files/62102364"
df = pd.read_csv(url)
df_raw = df.drop(columns=["Unnamed: 0"], errors="ignore")

# ============================================================
# 3) Build design matrix
# ============================================================
def build_design_matrix(df: pd.DataFrame, drop_cols=None):
    df = df.copy()

    df["Age_c"] = df["Age"] - 30.0

    irsd_dummies = pd.get_dummies(df["IRSD_quintile"], prefix="irsd", drop_first=False)
    if "irsd_5" in irsd_dummies.columns:
        irsd_dummies = irsd_dummies.drop(columns=["irsd_5"])

    smoke_dummies = pd.get_dummies(df["smoking_status"], prefix="smoke", drop_first=False)
    smoke_dummies = smoke_dummies.fillna(0)
    if "smoke_non" in smoke_dummies.columns:
        smoke_dummies = smoke_dummies.drop(columns=["smoke_non"])

    for col in ["AF", "CKD", "diabetes", "cvd_event"]:
        df[col] = df[col].astype(int)

    X = pd.concat(
        [
            df[["Age_c", "AF", "CKD", "diabetes", "HbA1c", "eGFR", "SBP"]],
            smoke_dummies,
            irsd_dummies,
        ],
        axis=1
    )

    cols_to_drop = []
    if drop_cols is None:
        events = df["cvd_event"].astype(bool)

        for col in X.columns:
            v_all      = X[col].var()
            v_event    = X.loc[events, col].var() if events.sum() > 0 else 0.0
            v_nonevent = X.loc[~events, col].var() if (~events).sum() > 0 else 0.0

            if (v_all < 1e-6) or (v_event < 1e-6) or (v_nonevent < 1e-6):
                cols_to_drop.append(col)
    else:
        cols_to_drop = drop_cols

    X_reduced = X.drop(columns=cols_to_drop, errors="ignore")
    return X_reduced, cols_to_drop


X_all, cols_to_drop = build_design_matrix(df_raw, drop_cols=None)

cox_df = pd.concat(
    [
        df_raw[["cvd_time", "cvd_event"]].reset_index(drop=True),
        X_all.reset_index(drop=True),
    ],
    axis=1
).dropna()

print("Final modelling dataframe shape:", cox_df.shape)
print("Dropped columns:", cols_to_drop)


# ============================================================
# 4) Fit two CoxPH models (FULL vs LASSO)
# ============================================================
PENALIZER = 0.01

cph_full = CoxPHFitter(penalizer=PENALIZER, l1_ratio=0.0)
cph_full.fit(cox_df, duration_col="cvd_time", event_col="cvd_event", show_progress=True)
print("\n=== CoxPH (FULL) ===")
cph_full.print_summary()

cph_lasso = CoxPHFitter(penalizer=PENALIZER, l1_ratio=0.95)  # LASSO-heavy elastic net
cph_lasso.fit(cox_df, duration_col="cvd_time", event_col="cvd_event", show_progress=True)
print("\n=== CoxPH (LASSO / Sparse) ===")
cph_lasso.print_summary()


# ============================================================
# 5) Predict absolute risk at a time horizon t0
# ============================================================
def predict_risk_at_horizon(cph: CoxPHFitter, df_covariates: pd.DataFrame, t0: float) -> pd.Series:
    # Linear predictor
    lp = cph.predict_log_partial_hazard(df_covariates).values.reshape(-1)

    # Baseline survival S0(t)
    base_surv = cph.baseline_survival_

    times = base_surv.index.values.astype(float)
    s0_values = base_surv.values.reshape(-1).astype(float)

    if t0 <= times.min():
        s0_t0 = float(s0_values[0])
    elif t0 >= times.max():
        s0_t0 = float(s0_values[-1])
    else:
        s0_t0 = float(np.interp(t0, times, s0_values))

    # Individual survival and risk
    surv_i = np.power(s0_t0, np.exp(lp))
    risk_i = 1.0 - surv_i
    return pd.Series(risk_i, index=df_covariates.index, name=f"risk_t{t0}")


# Choose horizon (e.g., 5 years if cvd_time is in years; adjust if months/days)
T0 = 5.0

# Covariate matrix only (drop duration/event columns)
cov_cols = [c for c in cox_df.columns if c not in ["cvd_time", "cvd_event"]]
X_cov = cox_df[cov_cols].copy()

risk_full  = predict_risk_at_horizon(cph_full,  X_cov, T0)
risk_lasso = predict_risk_at_horizon(cph_lasso, X_cov, T0)


# ============================================================
# 6) Percentile-NRI (distributional reclassification)
# ============================================================
def percentile_nri(
    time: pd.Series,
    event: pd.Series,
    risk_a: pd.Series,
    risk_b: pd.Series,
    t0: float,
    K: int = 10,
) -> dict:
    """
    Percentile-NRI comparing Model A vs Model B at horizon t0 using K equal-sized risk bins.

    Returns:
      - nri_event, nri_nonevent, nri_total
      - counts + movement tables
    """
    df = pd.DataFrame({
        "time": time.astype(float),
        "event": event.astype(int),
        "risk_a": risk_a.astype(float),
        "risk_b": risk_b.astype(float),
    })

    # Horizon status
    is_event_by_t0 = (df["event"] == 1) & (df["time"] <= t0)
    is_known_nonevent_by_t0 = (df["time"] >= t0)

    eligible = is_event_by_t0 | is_known_nonevent_by_t0
    df = df.loc[eligible].copy()

    df["status_t0"] = np.where(is_event_by_t0.loc[df.index], 1, 0)  # 1=event by t0, 0=non-event at t0

    # Create K equal-sized percentile bins based on each model's risk ranking
    # Use ranks to avoid ties causing qcut issues.
    df["rank_a"] = df["risk_a"].rank(method="average")
    df["rank_b"] = df["risk_b"].rank(method="average")

    df["bin_a"] = pd.qcut(df["rank_a"], q=K, labels=False, duplicates="drop")
    df["bin_b"] = pd.qcut(df["rank_b"], q=K, labels=False, duplicates="drop")

    # If duplicates="drop" reduced bins, update K_effective
    K_eff = int(max(df["bin_a"].max(), df["bin_b"].max()) + 1)

    # Movement direction
    df["move"] = np.where(df["bin_b"] > df["bin_a"], "up",
                   np.where(df["bin_b"] < df["bin_a"], "down", "same"))

    # Event and non-event subsets
    df_e = df[df["status_t0"] == 1]
    df_n = df[df["status_t0"] == 0]

    # Proportions
    Ue = (df_e["move"] == "up").mean()   if len(df_e) else np.nan
    De = (df_e["move"] == "down").mean() if len(df_e) else np.nan
    Un = (df_n["move"] == "up").mean()   if len(df_n) else np.nan
    Dn = (df_n["move"] == "down").mean() if len(df_n) else np.nan

    nri_event = Ue - De
    nri_nonevent = Dn - Un
    nri_total = nri_event + nri_nonevent

    tab_event = pd.crosstab(df_e["bin_a"], df_e["bin_b"], dropna=False)
    tab_nonev = pd.crosstab(df_n["bin_a"], df_n["bin_b"], dropna=False)

    return {
        "t0": t0,
        "K_requested": K,
        "K_effective": K_eff,
        "n_eligible": int(len(df)),
        "n_event_by_t0": int(len(df_e)),
        "n_nonevent_at_t0": int(len(df_n)),
        "Ue": float(Ue) if np.isfinite(Ue) else None,
        "De": float(De) if np.isfinite(De) else None,
        "Un": float(Un) if np.isfinite(Un) else None,
        "Dn": float(Dn) if np.isfinite(Dn) else None,
        "nri_event": float(nri_event) if np.isfinite(nri_event) else None,
        "nri_nonevent": float(nri_nonevent) if np.isfinite(nri_nonevent) else None,
        "nri_total": float(nri_total) if np.isfinite(nri_total) else None,
        "table_event": tab_event,
        "table_nonevent": tab_nonev,
        "eligible_df": df,  # keep for inspection
    }


# Run NRI: Model A = FULL, Model B = LASSO (so positive NRI means LASSO reclassifies better)
nri_out = percentile_nri(
    time=cox_df["cvd_time"],
    event=cox_df["cvd_event"],
    risk_a=risk_full,
    risk_b=risk_lasso,
    t0=T0,
    K=10
)

Final modelling dataframe shape: (50000, 15)
Dropped columns: []
Iteration 1: norm_delta = 1.69e+00, step_size = 0.9500, log_lik = -21528.74784, newton_decrement = 5.05e+03, seconds_since_start = 0.4
Iteration 2: norm_delta = 4.68e+00, step_size = 0.9500, log_lik = -22058.07755, newton_decrement = 1.01e+04, seconds_since_start = 1.1
Iteration 3: norm_delta = 1.26e+01, step_size = 0.9500, log_lik = -37014.23881, newton_decrement = 4.34e+04, seconds_since_start = 1.8
Iteration 4: norm_delta = 1.04e+00, step_size = 0.2327, log_lik = -19683.87064, newton_decrement = 1.09e+03, seconds_since_start = 2.5
Iteration 5: norm_delta = 5.58e-01, step_size = 0.2965, log_lik = -19143.73567, newton_decrement = 3.76e+02, seconds_since_start = 3.2
Iteration 6: norm_delta = 2.11e-01, step_size = 0.5011, log_lik = -18866.52901, newton_decrement = 6.78e+01, seconds_since_start = 3.8
Iteration 7: norm_delta = 2.22e-02, step_size = 0.8469, log_lik = -18801.25524, newton_decrement = 1.18e+00, seconds_since_st

<lifelines.CoxPHFitter: fitted with 50000 total observations, 47991 right-censored observations>
             duration col = 'cvd_time'
                event col = 'cvd_event'
                penalizer = 0.01
                 l1 ratio = 0.0
      baseline estimation = breslow
   number of observations = 50000
number of events observed = 2009
   partial log-likelihood = -18800.08
         time fit was run = 2026-03-04 02:47:45 UTC

---
               coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                      
Age_c          0.05      1.05      0.00            0.05            0.05                1.05                1.06
AF             1.32      3.76      0.13            1.06            1.58                2.90                4.88
CKD           -0.24      0.79      0.17           -0.57            0.09                0.57                1.10
diabetes       1.64      5.15      0.07            1.50            1.77                4.50                5.88
HbA1c          0.34      1.40      0.02            0.30            0.37                1.35                1.45
eGFR          -0.02      0.98      0.00           -0.03           -0.01                0.97                0.99
SBP            0.01      1.01      0.00            0.01            0.01                1.01                1.01
smoke_current  0.34      1.40      0.06            0.21            0.46                1.24                1.58
smoke_ex       0.28      1.33      0.05            0.18            0.38                1.20                1.47
irsd_1         0.02      1.02      0.05           -0.09            0.12                0.91                1.13
irsd_2         0.09      1.10      0.06           -0.03            0.21                0.98                1.23
irsd_3         0.10      1.11      0.05            0.00            0.21                1.00                1.23
irsd_4         0.01      1.01      0.06           -0.11            0.12                0.89                1.13

               cmp to     z      p  -log2(p)
covariate                                   
Age_c            0.00 28.51 <0.005    591.55
AF               0.00  9.98 <0.005     75.49
CKD              0.00 -1.41   0.16      2.66
diabetes         0.00 24.03 <0.005    421.30
HbA1c            0.00 17.20 <0.005    217.74
eGFR             0.00 -5.23 <0.005     22.53
SBP              0.00  8.90 <0.005     60.67
smoke_current    0.00  5.33 <0.005     23.30
smoke_ex         0.00  5.57 <0.005     25.27
irsd_1           0.00  0.27   0.78      0.35
irsd_2           0.00  1.54   0.12      3.01
irsd_3           0.00  1.97   0.05      4.35
irsd_4           0.00  0.10   0.92      0.12
---
Concordance = 0.87
Partial AIC = 37626.16
log-likelihood ratio test = 5457.34 on 13 df
-log2(p) of ll-ratio test = inf

Iteration 1: norm_delta = 1.79e+00, step_size = 0.9500, log_lik = -28113.64605, newton_decrement = 5.30e+03, seconds_since_start = 0.4
Iteration 2: norm_delta = 1.01e+01, step_size = 0.9500, log_lik = -27323.33986, newton_decrement = 1.89e+04, seconds_since_start = 0.8
Iteration 3: norm_delta = 9.18e+00, step_size = 0.2375, log_lik = -27172.59021, newton_decrement = 2.30e+04, seconds_since_start = 1.3
Iteration 4: norm_delta = 1.24e+00, step_size = 0.0756, log_lik = -23152.21189, newton_decrement = 1.81e+03, seconds_since_start = 1.7
Iteration 5: norm_delta = 8.49e-01, step_size = 0.1278, log_lik = -22088.05567, newton_decrement = 1.06e+03, seconds_since_start = 2.1
Iteration 6: norm_delta = 5.09e-01, step_size = 0.2160, log_lik = -21210.40495, newton_decrement = 4.86e+02, seconds_since_start = 2.6
Iteration 7: norm_delta = 2.84e-01, step_size = 0.3651, log_lik = -20572.45650, newton_decrement = 1.64e+02, seconds_since_start = 3.0
Iteration 8: norm_delta = 1.40e-01, step_size = 0.6170,

<lifelines.CoxPHFitter: fitted with 50000 total observations, 47991 right-censored observations>
             duration col = 'cvd_time'
                event col = 'cvd_event'
                penalizer = 0.01
                 l1 ratio = 0.95
      baseline estimation = breslow
   number of observations = 50000
number of events observed = 2009
   partial log-likelihood = -19425.77
         time fit was run = 2026-03-04 02:47:51 UTC

---
               coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                      
Age_c          0.05      1.05      0.00            0.05            0.06                1.05                1.06
AF             0.28      1.33      0.23           -0.16            0.73                0.85                2.07
CKD            0.00      1.00      0.00           -0.00            0.00                1.00                1.00
diabetes       1.80      6.05      0.08            1.65            1.95                5.18                7.05
HbA1c          0.27      1.31      0.02            0.22            0.31                1.25                1.37
eGFR          -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
SBP            0.00      1.00      0.00           -0.00            0.01                1.00                1.01
smoke_current  0.00      1.00      0.00           -0.00            0.00                1.00                1.00
smoke_ex       0.00      1.00      0.00           -0.00            0.00                1.00                1.00
irsd_1         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
irsd_2         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
irsd_3         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
irsd_4        -0.00      1.00      0.00           -0.00            0.00                1.00                1.00

               cmp to     z      p  -log2(p)
covariate                                   
Age_c            0.00 26.53 <0.005    512.86
AF               0.00  1.26   0.21      2.28
CKD              0.00  0.01   1.00      0.01
diabetes         0.00 22.93 <0.005    384.19
HbA1c            0.00 11.64 <0.005    101.60
eGFR             0.00 -0.01   1.00      0.01
SBP              0.00  1.81   0.07      3.83
smoke_current    0.00  0.01   1.00      0.01
smoke_ex         0.00  0.01   1.00      0.01
irsd_1           0.00  0.00   1.00      0.00
irsd_2           0.00  0.00   1.00      0.00
irsd_3           0.00  0.00   1.00      0.00
irsd_4           0.00 -0.00   1.00      0.00
---
Concordance = 0.87
Partial AIC = 38877.54
log-likelihood ratio test = 4205.96 on 13 df
-log2(p) of ll-ratio test = inf

In [7]:
print("\n=== Percentile-NRI (FULL -> LASSO) ===")
print(f"Horizon t0 = {nri_out['t0']}, bins K_effective = {nri_out['K_effective']}")
print(f"Eligible n = {nri_out['n_eligible']}")
print(f"Event by t0 n = {nri_out['n_event_by_t0']}")
print(f"Non-event at t0 n = {nri_out['n_nonevent_at_t0']}")
print(f"NRI_event    = {nri_out['nri_event']}")
print(f"NRI_nonevent = {nri_out['nri_nonevent']}")
print(f"NRI_total    = {nri_out['nri_total']}")


=== Percentile-NRI (FULL -> LASSO) ===
Horizon t0 = 5.0, bins K_effective = 10
Eligible n = 22458
Event by t0 n = 1941
Non-event at t0 n = 20517
NRI_event    = -0.008758371973209694
NRI_nonevent = -0.024759955159136315
NRI_total    = -0.03351832713234601


In [8]:
print("\nEvent movement table (bin_a -> bin_b):")
nri_out["table_event"]


Event movement table (bin_a -> bin_b):


bin_b,0,1,2,3,4,5,6,7,8,9
bin_a,,,,,,,,,,
0,9,5,0,0,0,0,0,0,0,0
1,5,9,7,1,1,0,0,0,0,0
2,2,5,13,7,2,0,0,0,0,0
3,0,4,8,16,17,4,0,0,0,0
4,0,0,5,17,16,17,3,0,0,0
5,0,0,1,6,12,21,25,3,0,0
6,0,0,0,0,3,17,42,25,0,0
7,0,0,0,0,0,7,35,69,43,0
8,0,0,0,1,0,2,3,31,202,18


In [9]:
print("\nNon-event movement table (bin_a -> bin_b):")
nri_out["table_nonevent"]


Non-event movement table (bin_a -> bin_b):


bin_b,0,1,2,3,4,5,6,7,8,9
bin_a,,,,,,,,,,
0,1800,419,13,0,0,0,0,0,0,0
1,380,1192,567,82,2,0,0,0,0,0
2,46,477,946,592,150,6,0,0,0,0
3,3,125,481,810,617,150,10,0,0,0
4,0,10,162,484,772,621,138,1,0,0
5,1,0,37,191,457,780,615,97,0,0
6,0,0,3,33,170,498,845,592,17,0
7,0,0,3,4,22,117,471,1027,448,0
8,0,0,0,1,5,6,58,397,1416,106
